In [1]:
# imports and where we keep things
import hashlib, io, json, urllib.request, zipfile, warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR, RESULTS, FIGURES = ROOT / "data" / "raw", ROOT / "results", ROOT / "figures"

In [2]:
# where the data comes from and the settings we fixed up front
ZIP_PATH = RAW_DIR / "heart+disease.zip"
UCI_URL = "https://archive.ics.uci.edu/static/public/45/heart+disease.zip"
UCI_ZIP_SHA256 = "b17cd273da9ce1caa4710fce80227ea454d4dbf9fcbc8e6a9121672751563adc"
FLAMBY_COMMIT = "edacf54d5211520583b0133d55ac39b6fda8324b"  # we copied flamby's split from fed_heart_disease/dataset.py

COLUMNS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"]
DROPPED = ["slope", "ca", "thal"]  # mostly missing outside cleveland, so we drop them like flamby does
CENTERS = {0: ("Cleveland", "processed.cleveland.data"), 1: ("Hungary", "processed.hungarian.data"),
           2: ("Switzerland", "processed.switzerland.data"), 3: ("VA Long Beach", "processed.va.data")}
CENTER_IDS = [0, 1, 2, 3]

# we scale with fixed limits instead of stats from the data, since those stats would leak patient info
BOUNDS = {"age": (0.0, 120.0), "trestbps": (0.0, 300.0), "chol": (0.0, 1000.0), "thalach": (0.0, 300.0), "oldpeak": (-10.0, 20.0)}
BINARY = ["sex", "fbs", "exang"]
CATEGORICAL = {"cp": [1, 2, 3, 4], "restecg": [0, 1, 2]}  # we drop the first level of each
CHOL_MISSING_FILL = 200.0  # only for our chol-as-missing check
FEATURES_MAIN = ["age", "sex", "trestbps", "chol", "fbs", "thalach", "exang", "oldpeak", "cp_2", "cp_3", "cp_4", "restecg_1", "restecg_2"]

In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def download(force=False):
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    if force or not ZIP_PATH.exists():
        with urllib.request.urlopen(UCI_URL) as r:
            ZIP_PATH.write_bytes(r.read())
    if sha256_file(ZIP_PATH) != UCI_ZIP_SHA256:
        raise ValueError("checksum mismatch, try download(force=True)")
    return ZIP_PATH


def load_raw():
    # loads all 920 rows. '?' means missing
    download()
    frames = []
    with zipfile.ZipFile(ZIP_PATH) as z:
        for cid, (name, fname) in CENTERS.items():
            df = pd.read_csv(io.StringIO(z.read(fname).decode("latin-1")), header=None, names=COLUMNS, na_values="?")
            df = df.apply(pd.to_numeric)
            df.insert(0, "row_in_file", np.arange(len(df)))
            df.insert(0, "center", name)
            df.insert(0, "center_id", cid)
            frames.append(df)
    return pd.concat(frames, ignore_index=True)


def flamby_filter_and_split(raw):
    # we follow flamby exactly so our numbers line up with their benchmark:
    # drop slope/ca/thal, drop rows still missing something, then split each hospital 66/34 (seed 43)
    out = []
    for cid in CENTER_IDS:
        df = raw[raw.center_id == cid].drop(columns=DROPPED).dropna(axis=0).reset_index(drop=True)
        y = (df["num"] > 0).astype(int)
        stratify = y if (y.nunique() > 1 and all((y == lev).sum() > 2 for lev in y.unique())) else None
        _, idx_test = train_test_split(np.arange(len(df)), test_size=1.0 - 0.66, train_size=0.66,
                                       random_state=43, shuffle=True, stratify=stratify)
        split = np.array(["train"] * len(df), dtype=object)
        split[idx_test] = "test"
        df["y"], df["split"], df["stratified"] = y.values, split, stratify is not None
        out.append(df)
    return pd.concat(out, ignore_index=True)

In [4]:
def _scale(v, lo, hi):
    return 2.0 * (np.clip(v, lo, hi) - lo) / (hi - lo) - 1.0


def transform_fixed(df, chol_policy="keep_zero"):
    # each row is handled on its own with fixed constants. "zero_as_missing" fills chol=0 and adds a flag
    cols, names = [], []
    for c in ["age", "sex", "trestbps", "chol", "fbs", "thalach", "exang", "oldpeak"]:
        v = df[c].to_numpy(dtype=float)
        if c in BINARY:
            assert np.isin(v, [0, 1]).all(), c
            cols.append(v)
        else:
            if c == "chol" and chol_policy == "zero_as_missing":
                v = np.where(v == 0, CHOL_MISSING_FILL, v)
            cols.append(_scale(v, *BOUNDS[c]))
        names.append(c)
    for c, vocab in CATEGORICAL.items():
        v = df[c].to_numpy()
        if not np.isin(v, vocab).all():
            raise ValueError(f"{c} has unknown values")
        for level in vocab[1:]:
            cols.append((v == level).astype(float))
            names.append(f"{c}_{level}")
    if chol_policy == "zero_as_missing":
        cols.append((df["chol"].to_numpy() == 0).astype(float))
        names.append("chol_missing")
    elif chol_policy != "keep_zero":
        raise ValueError(chol_policy)
    return np.column_stack(cols), names


def transform_flamby_parity(df, by_center=True):
    # flamby's own scaling uses the training data, so it isn't private. we only use it to compare
    X, names = transform_fixed(df)
    for c in BOUNDS:
        X[:, names.index(c)] = df[c].to_numpy(dtype=float)
    Z = X.copy()
    groups = df["center_id"].to_numpy() if by_center else np.zeros(len(df), dtype=int)
    is_train = (df["split"] == "train").to_numpy()
    for g in np.unique(groups):
        m = groups == g
        Z[m] = (X[m] - X[m & is_train].mean(0)) / (X[m & is_train].std(0, ddof=1) + 1e-9)
    return Z, names

In [5]:
@dataclass
class Site:
    cid: int
    name: str
    X_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    test_row_ids: np.ndarray  # like "0:12" (hospital:row), so we can match predictions across methods


def build_sites(retained, transform="fixed", chol_policy="keep_zero", train_fraction=None, subsample_seed=0):
    # train_fraction lets us shrink one hospital's training set for the size experiment. test sets never change
    if transform == "fixed":
        X, _ = transform_fixed(retained, chol_policy)
    elif transform == "flamby_center":
        X, _ = transform_flamby_parity(retained, by_center=True)
    elif transform == "flamby_pooled":
        X, _ = transform_flamby_parity(retained, by_center=False)
    else:
        raise ValueError(transform)
    ids = (retained.center_id.astype(str) + ":" + retained.row_in_file.astype(str)).to_numpy()
    y = retained.y.to_numpy().astype(float)
    sites = []
    for cid in CENTER_IDS:
        m = (retained.center_id == cid).to_numpy()
        idx_tr = np.flatnonzero(m & (retained.split == "train").to_numpy())
        te = m & (retained.split == "test").to_numpy()
        if train_fraction and train_fraction.get(cid, 1.0) < 1.0:
            rng = np.random.default_rng(subsample_seed)
            idx_tr = np.sort(rng.choice(idx_tr, size=int(round(train_fraction[cid] * len(idx_tr))), replace=False))
        sites.append(Site(cid, CENTERS[cid][0], X[idx_tr], y[idx_tr], X[te], y[te], ids[te]))
    return sites

In [6]:
raw = load_raw()
retained = flamby_filter_and_split(raw)
sites = build_sites(retained)
print(f"loaded {len(raw)} rows, kept {len(retained)} after filtering")

loaded 920 rows, kept 740 after filtering


In [7]:
# a quick look at the data before we train anything (also saved to results/audit/)
AUDIT = RESULTS / "audit"
AUDIT.mkdir(parents=True, exist_ok=True)
feat_cols = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak"]

counts = (retained.groupby(["center_id", "center", "split"]).size().unstack().rename(columns=str)
          .assign(retained=lambda d: d.sum(axis=1)))
counts.insert(0, "source", raw.groupby(["center_id", "center"]).size())
counts["removed"] = counts.source - counts.retained
counts["stratified_split"] = retained.groupby(["center_id", "center"]).stratified.first()

labels = retained.groupby(["center_id", "center", "split"]).y.agg(positive="sum", total="count")
labels["negative"] = labels.total - labels.positive
labels["prevalence"] = labels.positive / labels.total

missing = raw[[c for c in COLUMNS if c != "num"]].isna().groupby([raw.center_id, raw.center]).sum().T

unusual = pd.DataFrame({
    "chol_zero_source": raw.groupby("center").chol.apply(lambda s: int((s == 0).sum())),
    "chol_zero_retained": retained.groupby("center").chol.apply(lambda s: int((s == 0).sum())),
    "trestbps_zero_retained": retained.groupby("center").trestbps.apply(lambda s: int((s == 0).sum())),
    "oldpeak_negative_retained": retained.groupby("center").oldpeak.apply(lambda s: int((s < 0).sum())),
})

why = raw.drop(columns=DROPPED + ["center_id", "center", "row_in_file"]).isna()
removal = why[why.any(axis=1)].groupby(raw.loc[why.any(axis=1), "center"]).sum().T
removal = removal[(removal > 0).any(axis=1)]

X_all, names = transform_fixed(retained)
schema = pd.DataFrame({"feature": names,
                       "type": ["continuous" if n in BOUNDS else "binary" if n in BINARY else "one-hot" for n in names],
                       "min_after": X_all.min(0), "max_after": X_all.max(0)})

key = retained.center_id.astype(str) + ":" + retained.row_in_file.astype(str)
dupes = retained[retained.duplicated(["center_id"] + feat_cols, keep=False)]
checks = pd.DataFrame([
    ("four hospitals", sorted(retained.center_id.unique()) == CENTER_IDS),
    ("source rows", len(raw)),
    ("kept rows", len(retained)),
    ("each row in exactly one split", bool(key.is_unique)),
    ("label = (num > 0)", bool(((retained.num > 0).astype(int) == retained.y).all())),
    ("all features finite", bool(np.isfinite(X_all).all())),
    ("same 13 columns everywhere", all(s.X_train.shape[1] == 13 for s in sites)),
    ("test ids stay in their hospital", all(all(r.startswith(f"{s.cid}:") for r in s.test_row_ids) for s in sites)),
    ("identical feature rows within a hospital (kept)", int(retained.duplicated(["center_id"] + feat_cols).sum())),
    ("identical feature rows across hospitals", int(retained.drop_duplicates(["center_id"] + feat_cols).duplicated(feat_cols).sum())),
], columns=["check", "result"])

for name, t in [("counts", counts), ("labels", labels), ("missing_source", missing), ("unusual_values", unusual),
                ("removal_reasons", removal)]:
    t.to_csv(AUDIT / f"{name}.csv")
schema.to_csv(AUDIT / "feature_schema.csv", index=False)
checks.to_csv(AUDIT / "checks.csv", index=False)
retained[["center_id", "center", "row_in_file", "y", "split"]].to_csv(AUDIT / "split_manifest.csv", index=False)

In [8]:
# we should get 303/261/46/130 kept and 199/172/30/85 train, same as flamby
counts

,split,source,test,train,retained,removed,stratified_split
center_id,center,,,,,,
0,Cleveland,303,104,199,303,0,True
1,Hungary,294,89,172,261,33,True
2,Switzerland,123,16,30,46,77,False
3,VA Long Beach,200,45,85,130,70,True


In [9]:
# switzerland has no healthy patients in training and only one in test, so its scores are shaky
labels

positive  total  negative  prevalence
center_id center        split                                       
0         Cleveland     test         48    104        56    0.461538
                        train        91    199       108    0.457286
1         Hungary       test         33     89        56    0.370787
                        train        65    172       107    0.377907
2         Switzerland   test         15     16         1    0.937500
                        train        30     30         0    1.000000
3         VA Long Beach test         35     45        10    0.777778
                        train        66     85        19    0.776471

In [10]:
# how many values are missing in the original files
missing

center_id,0,1,2,3
center,Cleveland,Hungary,Switzerland,VA Long Beach
age,0,0,0,0
sex,0,0,0,0
cp,0,0,0,0
trestbps,0,1,2,56
chol,0,23,0,7
fbs,0,8,75,7
restecg,0,1,1,0
thalach,0,1,1,53
exang,0,1,1,53


In [11]:
# which missing fields caused rows to be dropped
removal

center,Hungary,Switzerland,VA Long Beach
trestbps,1,2,56
chol,23,0,7
fbs,8,75,7
restecg,1,1,0
thalach,1,1,53
exang,1,1,53
oldpeak,0,6,56


In [12]:
# chol = 0 really means it wasn't measured (all of switzerland, about a quarter of va)
unusual

,chol_zero_source,chol_zero_retained,trestbps_zero_retained,oldpeak_negative_retained
center,,,,
Cleveland,0,0,0,0
Hungary,0,0,0,0
Switzerland,123,46,0,1
VA Long Beach,49,33,1,1


In [13]:
schema

,feature,type,min_after,max_after
0,age,continuous,-0.533333,0.283333
1,sex,binary,0.000000,1.000000
2,trestbps,continuous,-1.000000,0.333333
3,chol,continuous,-1.000000,0.206000
4,fbs,binary,0.000000,1.000000
5,thalach,continuous,-0.600000,0.346667
6,exang,binary,0.000000,1.000000
7,oldpeak,continuous,-0.400000,0.080000
8,cp_2,one-hot,0.000000,1.000000
9,cp_3,one-hot,0.000000,1.000000


In [14]:
checks

,check,result
0,four hospitals,True
1,source rows,920
2,kept rows,740
3,each row in exactly one split,True
4,label = (num > 0),True
5,all features finite,True
6,same 13 columns everywhere,True
7,test ids stay in their hospital,True
8,identical feature rows within a hospital (kept),1
9,identical feature rows across hospitals,0


In [15]:
# one pair of identical rows at va. we kept both since they could be two different people
dupes[["center", "row_in_file", "split", "y"] + feat_cols]

,center,row_in_file,split,y,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak
701,VA Long Beach,139,train,1,58.0,1.0,3.0,150.0,219.0,0.0,1.0,118.0,1.0,0.0
729,VA Long Beach,187,test,1,58.0,1.0,3.0,150.0,219.0,0.0,1.0,118.0,1.0,0.0


In [16]:
# if any of these fail, the notebook stops
assert raw.groupby("center_id").size().to_dict() == {0: 303, 1: 294, 2: 123, 3: 200}
c = retained.groupby(["center_id", "split"]).size().unstack()
assert c["train"].to_dict() == {0: 199, 1: 172, 2: 30, 3: 85}
assert c["test"].to_dict() == {0: 104, 1: 89, 2: 16, 3: 45}
assert checks.set_index("check").loc[["four hospitals", "each row in exactly one split", "label = (num > 0)",
                                      "all features finite", "same 13 columns everywhere", "test ids stay in their hospital"], "result"].all()
assert names == FEATURES_MAIN

# changing one patient shouldn't change anyone else's features
changed = retained.copy()
changed.loc[0, ["age", "chol", "cp"]] = [99, 900, 4]
X2, _ = transform_fixed(changed)
assert np.array_equal(X_all[1:], X2[1:]) and not np.array_equal(X_all[0], X2[0])
print("all the data checks look good")

all the data checks look good
